# Business purpose — GB market and settlement primer

This notebook gives an energy portfolio operator a concise, worked view of Great Britain half-hourly settlement, contracted-versus-metered imbalance and the information advantage of probabilistic forecasts. It uses reusable GridMatch settlement utilities and deterministic examples; it does **not** reproduce a licensed supplier's complete settlement system.

## Participants and commercial context

- **Generators** produce physical electricity; **consumers** use it.
- **Suppliers** contract for and settle customer energy, while **NESO** operates the electricity system and **Elexon** administers Balancing and Settlement Code arrangements.
- Physical electricity flows through the grid; a commercial renewable match is an accounting/contractual allocation and does not create a dedicated physical wire.

An operator therefore needs coherent site and portfolio forecasts: metered demand or generation that differs from contracted volume creates an imbalance position.

In [1]:
import numpy as np
from IPython.display import display
from gridmatch.research.market import settlement_day_examples, imbalance_worked_example, forecast_information_example, save_market_artifacts

np.random.seed(20260725)
artifact_paths = save_market_artifacts()
display(settlement_day_examples())
print({name: str(path) for name, path in artifact_paths.items()})

,settlement_date,day_type,period_count,first_period_utc,first_period_london,last_period_utc,last_period_london
0,2025-01-15,normal,48,2025-01-15T00:00:00+00:00,2025-01-15T00:00:00+00:00,2025-01-15T23:30:00+00:00,2025-01-15T23:30:00+00:00
1,2025-03-30,spring clock change,46,2025-03-30T00:00:00+00:00,2025-03-30T00:00:00+00:00,2025-03-30T22:30:00+00:00,2025-03-30T23:30:00+01:00
2,2025-10-26,autumn clock change,50,2025-10-25T23:00:00+00:00,2025-10-26T00:00:00+01:00,2025-10-26T23:30:00+00:00,2025-10-26T23:30:00+00:00


{'settlement_table': 'artifacts\\tables\\00_settlement_day_examples.csv', 'imbalance_table': 'artifacts\\tables\\00_imbalance_worked_example.csv', 'forecast_table': 'artifacts\\tables\\00_point_vs_probabilistic.csv', 'settlement_figure': 'artifacts\\figures\\00_gb_settlement_period_counts.png'}


## Settlement-day mechanics

A normal local delivery day has 48 half-hours. The spring clock change removes one local hour (46 periods); the autumn change repeats one local hour (50 periods). UTC timestamps remain unambiguous, while Europe/London labels provide the operational display. Forecast records must keep **issue time** (what was knowable) separate from **valid/delivery time** (what is being predicted).

In [2]:
imbalance = imbalance_worked_example()
display(imbalance)
print(f"Net example imbalance: {imbalance['imbalance_mwh'].sum():.3f} MWh")

,settlement_period,contracted_mwh,metered_mwh,imbalance_mwh,position
0,17,0.80,0.77,-0.03,short
1,18,0.82,0.88,0.06,long
2,19,0.85,0.91,0.06,long
3,20,0.86,0.83,-0.03,short


Net example imbalance: 0.060 MWh


## Point versus probabilistic forecasts

A point forecast supports a central contracted volume. Calibrated q10/q50/q90 estimates additionally expose plausible tails, enabling an operator to compare short and long risk when costs are asymmetric. Quantiles are information, not an instruction to trade; calibration and governance remain necessary.

In [3]:
display(forecast_information_example())

,forecast_type,published_values,operator_use,limitation
0,point,single expected MWh,planning central volume,does not express tail exposure
1,probabilistic,"q10, q50, q90 MWh",size risk buffers and compare asymmetric costs,requires calibration and explicit issue time


## Findings, limitations and production implications

**Findings:** the reusable functions produce 48/46/50 periods and unambiguous UTC boundaries; the worked imbalance convention is metered minus contracted volume, where positive is long and negative is short. **Limitations:** worked volumes are deterministic illustrations, not public or simulated customer observations, and omit the full BSC settlement calculation. **Production implications:** preserve issue time, delivery time, contract volume and meter volume; test DST days explicitly; only use probabilistic forecasts after calibration and human-approved risk policy.